
# Python for Automotive Engineers — **File Handling** (with Notes, Examples, Exercises, Answers & Auto‑Checks)

This hands‑on notebook teaches **Python file I/O** end‑to‑end using **automotive SDLC** data:
- CAN/UDS log lines (text)
- ECU version maps (CSV)
- Calibration artifacts (JSON)
- Binary payloads (bytes/struct)
- Large log processing (streaming, memory‑mapping)
- Paths, globbing, atomic writes, temp files, compression, encodings & errors

> ✅ Includes **explanatory notes**, **runnable examples**, **exercises with collapsible answers**, and **pytest cells** to auto‑check your solutions.

---



## 1) Paths & Context Managers
- Prefer **`pathlib.Path`** for OS‑independent path code.
- Use **context managers** (`with open(...) as f:`) so files always close, even on exceptions.


In [ ]:

from pathlib import Path
root = Path.cwd()
example_dir = root / "_demo_files"
example_dir.mkdir(exist_ok=True)
example_dir


In [ ]:

# Create a small demo file safely
sample = example_dir / "note.txt"
with sample.open("w", encoding="utf-8", newline="
") as f:
    f.write("Vehicle: TestCar
")
    f.write("ECU: PCM
")

# Read it back
with sample.open("r", encoding="utf-8") as f:
    print(f.read())



## 2) Encodings & Newlines
- Use `encoding='utf-8'` for portability.
- Control newlines with `newline='
'` when writing text.
- On reading, Python universal newlines normalize line endings.


In [ ]:

# Demonstrate encoding errors and strategies
bad_bytes = b"Calib: ÿþý"  # invalid in UTF-8
binary = example_dir / "weird.bin"
binary.write_bytes(bad_bytes)

try:
    # This will fail
    print(binary.read_text(encoding="utf-8"))
except UnicodeDecodeError as e:
    print("UnicodeDecodeError:", e.__class__.__name__, e)

# Robust read with replacement
print(binary.read_text(encoding="utf-8", errors="replace"))



## 3) Reading Line‑by‑Line (Streaming Large Files)
- Iterate the file object to **stream** lines without loading everything into memory.
- Useful for long CAN logs.


In [ ]:

log_path = example_dir / "can.log"
log_lines = [
    "1627561001,0x200,8,11 22 33 44 55 66 77 88
",
    "1627561002,0x321,3,01 02 03
",
    "1627561003,0x7E8,8,62 F1 90 4D 41 31 58 59
",
]
log_path.write_text("".join(log_lines), encoding="utf-8")

# Stream read
with log_path.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        print(i, line.strip())



## 4) Parsing a CAN Log Line (Text → Dict)
Each line format: `timestamp,can_id_hex,dlc,data_bytes`  
Example: `1627561001,0x200,8,11 22 33 44 55 66 77 88`


In [ ]:

from typing import Dict, Any

def parse_can_line(line: str) -> Dict[str, Any]:
    ts_s, can_hex, dlc_s, data_s = line.strip().split(",", 3)
    return {
        "ts": int(ts_s),
        "id": int(can_hex, 16),
        "dlc": int(dlc_s),
        "data": bytes(int(b, 16) for b in data_s.split())
    }

example = parse_can_line(log_lines[0])
example, example["data"].hex()



## 5) CSV: ECU Version Maps
- Use the built‑in **`csv`** module for simple CSV.
- For heavy analytics, you might use pandas (not required here).


In [ ]:

import csv
versions_csv = example_dir / "ecu_versions.csv"
rows = [
    {"ecu": "PCM", "version": "v3.4.2"},
    {"ecu": "ABS", "version": "v1.9.0"},
]

with versions_csv.open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["ecu", "version"]) 
    w.writeheader()
    w.writerows(rows)

with versions_csv.open("r", encoding="utf-8") as f:
    r = csv.DictReader(f)
    print(list(r))



## 6) JSON: Calibration Artifacts
- Use **`json.load`/`json.dump`** for configs and calibration params.
- Keep `indent=2`, `sort_keys=True` for readable diffs.


In [ ]:

import json
calib_json = example_dir / "calibration.json"
calib = {
    "TPS_OFFSET": 0.02,
    "RPM_LIMITS": [700, 6500],
    "SENSORS": {"speed": "km/h", "rpm": "min^-1"}
}
calib_json.write_text(json.dumps(calib, indent=2, sort_keys=True), encoding="utf-8")

loaded = json.loads(calib_json.read_text(encoding="utf-8"))
loaded



## 7) Binary Files & `struct`
- For signal capture payloads, write/read **bytes**.
- Use **`struct`** for fixed‑layout binary records.


In [ ]:

import struct
bin_file = example_dir / "payload.bin"
# record: uint32 timestamp, uint16 id, uint8 dlc, 8 bytes data
record = struct.pack(">I H B 8s", 1627561001, 0x200, 8, bytes.fromhex("1122334455667788"))
bin_file.write_bytes(record)

# read back
(ts, cid, dlc, data) = struct.unpack(">I H B 8s", bin_file.read_bytes())
ts, hex(cid), dlc, data.hex()



## 8) Safe Writes (Atomic), Temp Files, Compression
- **Atomic write**: write to a temp file, then replace → no partial files.
- **`tempfile`** creates safe temporary files/dirs.
- **Compression**: `gzip`, `zipfile` for space‑saving archives.


In [ ]:

import os, tempfile, gzip
from pathlib import Path

safe_out = example_dir / "safe_output.txt"
text = "This is a critical report.
"

# Atomic write pattern
with tempfile.NamedTemporaryFile("w", delete=False, dir=example_dir, encoding="utf-8", newline="
") as tf:
    tf.write(text)
    tmp_name = tf.name
os.replace(tmp_name, safe_out)  # atomic move on same filesystem

# Gzip compression
gz_path = example_dir / "can.log.gz"
with gzip.open(gz_path, "wt", encoding="utf-8") as gz:
    gz.write("".join(log_lines))

safe_out.exists(), gz_path.exists()



## 9) Memory‑Mapped Files (mmap) — Fast Searches
For very large logs, **`mmap`** lets you scan without reading everything into Python first.


In [ ]:

import mmap

needle = b"0x200"
with log_path.open("rb") as f, mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ) as mm:
    count = 0
    pos = 0
    while True:
        pos = mm.find(needle, pos)
        if pos == -1:
            break
        count += 1
        pos += 1
    count



## 10) Finding Files: `glob` & `pathlib.rglob`
Find logs/calibration files by patterns.


In [ ]:

list(example_dir.rglob("*.log")), list(example_dir.glob("*.json"))



## 11) Exceptions & Logging
Always catch the **narrowest** exceptions; log context. For quick demos, `print` is fine; in production use `logging`.


In [ ]:

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s:%(message)s")

try:
    Path("missing.txt").read_text()
except FileNotFoundError:
    logging.warning("missing.txt not found — continuing")



---

## Exercises (Implement the functions below)

Implement the following and then run the **pytest** cell to auto‑check:
1. `parse_can_line_ex(line)` — same as earlier but return hex string for `id` in `"0x..."` and the `data` as a list of ints.
2. `load_can_log(path)` — parse every non‑empty line into dicts using `parse_can_line_ex`.
3. `save_versions_csv(path, mapping)` — write `{ecu: version}` to CSV with header `ecu,version`.
4. `load_versions_csv(path)` — read the same CSV back into dict.
5. `read_calibration_json(path)` / `write_calibration_json(path, obj)` — JSON helpers with `indent=2`.
6. `atomic_write_text(path, text)` — atomic write using a temp file + `os.replace`.
7. `count_keyword(path, keyword)` — case‑sensitive count in a text file.
8. `tail_file(path, n)` — return the last `n` lines efficiently.
9. `find_files(root, pattern)` — return sorted list of file paths (strings) matching a glob pattern under `root`.
10. `checksum_file(path)` — return **sha256** hex digest of file bytes.


In [ ]:

# ===== Student Solutions =====
from pathlib import Path
import os, csv, json, tempfile, hashlib

def parse_can_line_ex(line: str):
    """Parse 'ts,0xID,dlc,hex bytes'. Return dict with id as hex string and data as list[int]."""
    raise NotImplementedError


def load_can_log(path):
    """Return list of parsed dicts for each non-empty line."""
    raise NotImplementedError


def save_versions_csv(path, mapping: dict):
    """Write mapping {ecu:version} to CSV with header ecu,version."""
    raise NotImplementedError


def load_versions_csv(path):
    """Read ecu_versions CSV back into dict."""
    raise NotImplementedError


def read_calibration_json(path):
    """Read JSON from path and return dict."""
    raise NotImplementedError


def write_calibration_json(path, obj):
    """Write dict as pretty JSON (indent=2, sort_keys=True)."""
    raise NotImplementedError


def atomic_write_text(path, text: str):
    """Atomically write text to path using temp file + os.replace."""
    raise NotImplementedError


def count_keyword(path, keyword: str) -> int:
    """Return count of keyword occurrences (case-sensitive) in a text file."""
    raise NotImplementedError


def tail_file(path, n: int):
    """Return the last n lines as a list of strings without trailing newlines."""
    raise NotImplementedError


def find_files(root, pattern: str):
    """Return sorted list[str] of file paths matching pattern under root (recursive)."""
    raise NotImplementedError


def checksum_file(path) -> str:
    """Return sha256 hex digest of the file at path."""
    raise NotImplementedError



<details><summary><strong>Answer Key — Click to expand</strong></summary>

```python
from pathlib import Path
import os, csv, json, tempfile, hashlib

def parse_can_line_ex(line: str):
    ts_s, can_hex, dlc_s, data_s = line.strip().split(',', 3)
    return {
        'ts': int(ts_s),
        'id': can_hex.lower(),
        'dlc': int(dlc_s),
        'data': [int(b, 16) for b in data_s.split()]
    }


def load_can_log(path):
    out = []
    p = Path(path)
    with p.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            out.append(parse_can_line_ex(line))
    return out


def save_versions_csv(path, mapping: dict):
    p = Path(path)
    with p.open('w', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        w.writerow(['ecu','version'])
        for k, v in mapping.items():
            w.writerow([k, v])


def load_versions_csv(path):
    p = Path(path)
    out = {}
    with p.open('r', encoding='utf-8') as f:
        r = csv.DictReader(f)
        for row in r:
            out[row['ecu']] = row['version']
    return out


def read_calibration_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def write_calibration_json(path, obj):
    Path(path).write_text(json.dumps(obj, indent=2, sort_keys=True), encoding='utf-8')


def atomic_write_text(path, text: str):
    p = Path(path)
    with tempfile.NamedTemporaryFile('w', delete=False, dir=str(p.parent), encoding='utf-8', newline='
') as tf:
        tf.write(text)
        tmp = tf.name
    os.replace(tmp, p)


def count_keyword(path, keyword: str) -> int:
    count = 0
    with open(path, 'r', encoding='utf-8') as f:
        for chunk in f:
            count += chunk.count(keyword)
    return count


def tail_file(path, n: int):
    # Efficient: read blocks from end
    p = Path(path)
    size = p.stat().st_size
    with p.open('rb') as f:
        block = -1
        data = []
        while len(data) <= n and abs(block)*1024 < size + 1024:
            try:
                f.seek(block*1024, os.SEEK_END)
            except OSError:
                f.seek(0)
            data.insert(0, f.read(1024))
            block -= 1
    text = b''.join(data).decode('utf-8', errors='replace').splitlines()
    return text[-n:]


def find_files(root, pattern: str):
    return sorted(str(p) for p in Path(root).rglob(pattern))


def checksum_file(path) -> str:
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()
```

</details>


In [ ]:

import os, json, csv, tempfile, hashlib
from pathlib import Path


def make_log(tmp: Path) -> Path:
    content = "
".join([
        "1627561001,0x200,8,11 22 33 44 55 66 77 88",
        "1627561002,0x321,3,01 02 03",
        "",
        "1627561003,0x7E8,8,62 F1 90 4D 41 31 58 59",
    ]) + "
"
    p = tmp / 'log.txt'
    p.write_text(content, encoding='utf-8', newline='
')
    return p


def test_parse_can_line_ex():
    d = parse_can_line_ex("1627561002,0x321,3,01 02 03")
    assert d['ts'] == 1627561002 and d['id'] == '0x321' and d['dlc'] == 3 and d['data'] == [1,2,3]


def test_load_can_log(tmp_path):
    p = make_log(tmp_path)
    out = load_can_log(p)
    assert len(out) == 3 and out[0]['id'] == '0x200'


def test_versions_csv(tmp_path):
    mapping = {'PCM':'v3.4.2','ABS':'v1.9.0'}
    csvp = tmp_path / 'v.csv'
    save_versions_csv(csvp, mapping)
    back = load_versions_csv(csvp)
    assert back == mapping


def test_json_helpers(tmp_path):
    obj = {'TPS_OFFSET':0.02,'RPM_LIMITS':[700,6500]}
    jp = tmp_path / 'c.json'
    write_calibration_json(jp, obj)
    out = read_calibration_json(jp)
    assert out == obj


def test_atomic_and_counts(tmp_path):
    tp = tmp_path / 'a.txt'
    atomic_write_text(tp, 'hello
hello world
')
    assert tp.exists()
    assert count_keyword(tp, 'hello') == 2


def test_tail_and_find_and_checksum(tmp_path):
    # tail
    p = tmp_path / 'x.txt'
    p.write_text('
'.join(str(i) for i in range(100)), encoding='utf-8')
    assert tail_file(p, 3) == ['97','98','99']
    # find files
    (tmp_path / 'dir').mkdir()
    (tmp_path / 'dir' / 'file.log').write_text('x', encoding='utf-8')
    found = find_files(tmp_path, '*.log')
    assert any(s.endswith('file.log') for s in found)
    # checksum
    h = checksum_file(p)
    assert isinstance(h, str) and len(h) == 64


In [ ]:

# Run tests (requires pytest)
try:
    import pytest
    pytest.main(["-q"])  # or run !pytest -q
except Exception as e:
    print("Install pytest first: pip install pytest
", e)
